<a href="https://colab.research.google.com/github/eltongaspar/python/blob/Advpl/Aula_18_Integracao_e_APIs_Aluno_Colab_ngrok_ProEducador.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 18 – Integração e APIs
## Deep Learning e Inteligência Artificial Generativa Aplicada

### Capacidades
- Utilizar frameworks de Deep Learning.

### Conhecimentos
- APIs para modelos de IA.
- Fluxo de inferência.
- Integração empresarial.

### Estratégia
Demonstração de integração simples entre modelo e aplicação.

### Recursos
- FastAPI.
- Colab.
- Python.

### Critérios de avaliação
- Estrutura fluxo de inferência corretamente.
- Integra modelo à aplicação proposta.
- Documenta processo técnico.

### Evidência de aprendizagem
API funcional.

### Versão do Aluno — com campos para preenchimento

## 1. Contextualização

Nesta aula, vamos transformar um modelo de IA em um serviço acessível por uma aplicação.

A proposta é criar uma API com **FastAPI** para receber dados de sensores industriais e retornar uma predição de falha.

Fluxo principal:

1. treinar ou carregar um modelo;
2. salvar modelo, scaler e metadados;
3. testar uma função local de inferência;
4. criar uma API com FastAPI;
5. implementar endpoint `/predict`;
6. testar a API com requisição HTTP;
7. documentar tecnicamente o processo.

## 2. Conceitos fundamentais

### API

Uma API permite que sistemas diferentes se comuniquem. Em IA, ela permite que uma aplicação use um modelo treinado sem precisar conhecer os detalhes internos do treinamento.

### Inferência

Inferência é o uso do modelo já treinado para gerar uma predição sobre novos dados.

### Integração empresarial

Em uma empresa, a API pode ser chamada por sistemas como supervisórios, dashboards, aplicativos, ERPs, sistemas de manutenção ou plataformas web.

## 3. Fluxo de inferência recomendado

1. Receber dados em JSON.
2. Validar tipos e campos obrigatórios.
3. Organizar os dados na mesma ordem usada no treinamento.
4. Aplicar o mesmo pré-processamento, como scaler.
5. Executar a predição.
6. Interpretar a probabilidade.
7. Retornar JSON com classe, probabilidade e recomendação.

Um erro muito comum é esquecer o scaler usado no treinamento. Isso pode distorcer a entrada e gerar uma predição ruim.

In [54]:
# Instalação opcional no Colab
!pip install -q fastapi uvicorn nest-asyncio pyngrok scikit-learn pandas numpy joblib requests

print("Instalação preparada. Execute somente se necessário.")

Instalação preparada. Execute somente se necessário.


In [55]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 4. Carregamento do dataset

Faça upload do arquivo `aula18_dataset_requisicoes_inferencia.csv` e carregue o dataset.

In [56]:
import pandas as pd

# TODO: carregue o dataset
df = pd.read_csv("/content/drive/MyDrive/Classroom/Deep Learning Pró Educador/aula18_dataset_requisicoes_inferencia.csv")

# TODO: visualize as primeiras linhas
df.head()

,id_requisicao,temperatura_motor,vibracao_motor,corrente_motor,pressao_sistema,tempo_ciclo,falha_esperada_regra
0,1,68.78,4.47,6.86,5.72,31.52,0
1,2,62.85,1.79,7.77,5.14,48.86,0
2,3,58.52,2.09,5.98,3.79,56.39,1
3,4,42.29,3.08,8.89,5.09,30.27,0
4,5,66.93,4.58,10.03,3.98,43.60,1


In [57]:
# TODO: verifique informações gerais
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120 entries, 0 to 119
Data columns (total 7 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id_requisicao         120 non-null    int64  
 1   temperatura_motor     120 non-null    float64
 2   vibracao_motor        120 non-null    float64
 3   corrente_motor        120 non-null    float64
 4   pressao_sistema       120 non-null    float64
 5   tempo_ciclo           120 non-null    float64
 6   falha_esperada_regra  120 non-null    int64  
dtypes: float64(5), int64(2)
memory usage: 6.7 KB


In [58]:
# TODO: verifique distribuição da variável alvo
df["falha_esperada_regra"].value_counts(normalize=True)

,proportion
falha_esperada_regra,
0,0.783333
1,0.216667


## 5. Treinamento rápido do modelo

Complete as features, alvo, scaler e modelo.

In [59]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report

features = [
    "temperatura_motor",
    "vibracao_motor",
    "corrente_motor",
    "pressao_sistema",
    "tempo_ciclo"
]

target = "falha_esperada_regra"

X = df[features].values
y = df[target].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

modelo = MLPClassifier(hidden_layer_sizes=(16, 8), activation="relu", max_iter=800, random_state=42)

modelo.fit(X_train_scaled, y_train)
y_pred = modelo.predict(X_test_scaled)

print("Acurácia:", round(accuracy_score(y_test, y_pred), 4))
print(classification_report(y_test, y_pred))

Acurácia: 0.8333
              precision    recall  f1-score   support

           0       0.85      0.96      0.90        24
           1       0.67      0.33      0.44         6

    accuracy                           0.83        30
   macro avg       0.76      0.65      0.67        30
weighted avg       0.81      0.83      0.81        30



/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (800) reached and the optimization hasn't converged yet.
  warnings.warn(


## 6. Salvamento dos artefatos

Complete o salvamento do modelo, scaler e metadados.

In [60]:
import os
import joblib
import json

os.makedirs("artefatos_modelo", exist_ok=True)

joblib.dump(modelo, "artefatos_modelo/modelo_mlp_falhas.joblib")
joblib.dump(scaler, "artefatos_modelo/scaler_standard_falhas.joblib")

metadados = {
    "nome_modelo": "MLPClassifier - Falhas Industriais",
    "features": features,
    "target": target,
    "versao": "1.0",
    "descricao": "Modelo demonstrativo para inferência via FastAPI.",
    "limiar_falha": 0.5
}

with open("artefatos_modelo/metadados_modelo.json", "w", encoding="utf-8") as f:
    json.dump(metadados, f, ensure_ascii=False, indent=2)

print("Artefatos salvos com sucesso.")

Artefatos salvos com sucesso.


In [61]:
# TODO: liste os arquivos salvos
print(os.listdir("artefatos_modelo"))

['modelo_mlp_falhas.joblib', 'metadados_modelo.json', 'modelo_mlp.joblib', 'scaler_standard_falhas.joblib', 'scaler_standard.joblib']


## 7. Teste local do fluxo de inferência

Complete o carregamento dos artefatos e a aplicação do scaler.

In [62]:
modelo_carregado = joblib.load("artefatos_modelo/modelo_mlp_falhas.joblib")
scaler_carregado = joblib.load("artefatos_modelo/scaler_standard_falhas.joblib")

with open("artefatos_modelo/metadados_modelo.json", "r", encoding="utf-8") as f:
    metadados_carregados = json.load(f)

def inferir_falha(dados):
    entrada = pd.DataFrame([dados])
    entrada = entrada[metadados_carregados["features"]]
    entrada_scaled = scaler_carregado.transform(entrada.values)

    probabilidade = modelo_carregado.predict_proba(entrada_scaled)[0][1]
    classe = int(probabilidade >= metadados_carregados["limiar_falha"])

    rotulo = "falha" if classe == 1 else "normal"
    recomendacao = "Acionar equipe de manutenção." if classe == 1 else "Manter monitoramento normal."

    return {
        "classe_prevista": classe,
        "rotulo": rotulo,
        "probabilidade_falha": round(float(probabilidade), 4),
        "recomendacao": recomendacao
    }

exemplo = {
    "temperatura_motor": 78.0,
    "vibracao_motor": 4.5,
    "corrente_motor": 9.8,
    "pressao_sistema": 4.1,
    "tempo_ciclo": 55.0
}

inferir_falha(exemplo)

{'classe_prevista': 1,
 'rotulo': 'falha',
 'probabilidade_falha': 0.9923,
 'recomendacao': 'Acionar equipe de manutenção.'}

## 8. Criação do arquivo `app.py`

Complete os nomes dos arquivos do modelo e do scaler no código da API.

In [63]:
app_code = """
from fastapi import FastAPI
from pydantic import BaseModel, Field
import pandas as pd
import joblib
import json

app = FastAPI(
    title="API de Inferência - Falhas Industriais",
    description="API demonstrativa para predição de falhas usando modelo de IA.",
    version="1.0"
)

modelo = joblib.load("artefatos_modelo/modelo_mlp_falhas.joblib")
scaler = joblib.load("artefatos_modelo/scaler_standard_falhas.joblib")

with open("artefatos_modelo/metadados_modelo.json", "r", encoding="utf-8") as f:
    metadados = json.load(f)

class EntradaSensores(BaseModel):
    temperatura_motor: float = Field(..., description="Temperatura do motor em graus Celsius")
    vibracao_motor: float = Field(..., description="Vibração do motor")
    corrente_motor: float = Field(..., description="Corrente elétrica do motor")
    pressao_sistema: float = Field(..., description="Pressão do sistema")
    tempo_ciclo: float = Field(..., description="Tempo de ciclo do processo")

@app.get("/")
def raiz():
    return {"mensagem": "API de inferência ativa"}

@app.get("/health")
def health():
    return {"status": "ok"}

@app.get("/metadata")
def metadata():
    return metadados

@app.post("/predict")
def predict(entrada: EntradaSensores):
    dados = entrada.model_dump()
    df_entrada = pd.DataFrame([dados])
    df_entrada = df_entrada[metadados["features"]]
    entrada_scaled = scaler.transform(df_entrada.values)
    probabilidade = modelo.predict_proba(entrada_scaled)[0][1]
    classe = int(probabilidade >= metadados["limiar_falha"])
    rotulo = "falha" if classe == 1 else "normal"
    recomendacao = "Acionar equipe de manutenção." if classe == 1 else "Manter monitoramento normal."
    return {
        "classe_prevista": classe,
        "rotulo": rotulo,
        "probabilidade_falha": round(float(probabilidade), 4),
        "recomendacao": recomendacao,
        "entrada_recebida": dados
    }
"""

with open("app.py", "w", encoding="utf-8") as f:
    f.write(app_code)

print("Arquivo app.py criado com sucesso.")

Arquivo app.py criado com sucesso.


## 9. Execução da API no Google Colab com ngrok

Nesta etapa, a API FastAPI será executada diretamente no Google Colab e exposta por uma URL pública temporária do **ngrok**.

### Fluxo da execução

1. Instalar `fastapi`, `uvicorn`, `nest-asyncio`, `pyngrok` e `requests`.
2. Informar o token do ngrok, se necessário.
3. Encerrar túneis antigos.
4. Criar túnel público para a porta `8000`.
5. Iniciar o FastAPI em uma thread separada.
6. Acessar a documentação da API em `/docs`.
7. Testar os endpoints `/health` e `/predict`.

### Atenção

A URL do ngrok é temporária. Se a sessão do Colab for encerrada ou reiniciada, será necessário executar as células novamente.

In [64]:
# ============================================================
# 9.1 Instalação das bibliotecas para FastAPI + ngrok no Colab
# ============================================================

!pip install -q fastapi uvicorn nest-asyncio pyngrok requests

print("Bibliotecas instaladas/carregadas para execução da API no Colab.")

Bibliotecas instaladas/carregadas para execução da API no Colab.


In [65]:
# ============================================================
# 9.2 Configuração do token do ngrok
# ============================================================
# Crie uma conta no ngrok e copie seu Authtoken.
# Cole o token abaixo apenas em ambiente de teste individual.
# Não compartilhe notebooks com token preenchido.

from google.colab import userdata

# cole aqui seu token do ngrok, se necessário
NGROK_AUTHTOKEN = userdata.get('NGROK_API_KEY')

from pyngrok import ngrok

if NGROK_AUTHTOKEN.strip() != "":
    ngrok.set_auth_token(NGROK_AUTHTOKEN)
    print("Token do ngrok configurado com sucesso.")
else:
    print("Token do ngrok não informado.")
    print("Se sua conta exigir autenticação, informe o token em NGROK_AUTHTOKEN antes de continuar.")

Token do ngrok configurado com sucesso.


In [66]:
# ============================================================
# 9.3 Encerrar túneis antigos do ngrok
# ============================================================

from pyngrok import ngrok

try:
    ngrok.kill()
    print("Túneis ngrok anteriores encerrados.")
except Exception as e:
    print("Nenhum túnel anterior para encerrar ou ocorreu um aviso:", e)

Túneis ngrok anteriores encerrados.


In [67]:
# ============================================================
# 9.4 Rodar FastAPI no Colab e gerar URL pública com ngrok
# ============================================================

import nest_asyncio
import uvicorn
import threading
import time
from pyngrok import ngrok

nest_asyncio.apply()

PORTA_API = 8000

# Ensure the authtoken is stripped before use
ngrok.set_auth_token(NGROK_AUTHTOKEN.strip())

# Cria túnel público para a porta 8000
public_url = ngrok.connect(PORTA_API, bind_tls=True)
public_url = str(public_url).replace('NgrokTunnel: ', '').split(' -> ')[0]

print("URL pública da API:")
print(public_url)

print("\nAcesse a documentação automática da API em:")
print(f"{public_url}/docs")

print("\nEndpoint de predição:")
print(f"{public_url}/predict")

def iniciar_api():
    uvicorn.run("app:app", host="0.0.0.0", port=PORTA_API, log_level="info")

thread_api = threading.Thread(target=iniciar_api, daemon=True)
thread_api.start()

time.sleep(5)

print("\nServidor FastAPI iniciado em segundo plano no Colab.")

URL pública da API:
"https://neatly-conclude-cottage.ngrok-free.dev"

Acesse a documentação automática da API em:
"https://neatly-conclude-cottage.ngrok-free.dev"/docs

Endpoint de predição:
"https://neatly-conclude-cottage.ngrok-free.dev"/predict


INFO:     Started server process [6417]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)



Servidor FastAPI iniciado em segundo plano no Colab.


In [68]:
# ============================================================
# 9.5 Testar a API pela URL pública do ngrok
# ============================================================

import requests

# Força a remoção de aspas extras que estejam guardadas na variável
url_limpa = str(public_url).strip('"')

url_health = f"{url_limpa}/health"
resposta_health = requests.get(url_health)
print("/health status:", resposta_health.status_code)
print(resposta_health.json())

# Teste do endpoint /predict
url_predict = f"{url_limpa}/predict"
payload = {
    "temperatura_motor": 78.0,
    "vibracao_motor": 4.5,
    "corrente_motor": 9.8,
    "pressao_sistema": 4.1,
    "tempo_ciclo": 55.0
}
resposta_predict = requests.post(url_predict, json=payload)
print("\n/predict status:", resposta_predict.status_code)
print(resposta_predict.json())

INFO:     136.107.230.182:0 - "GET /health HTTP/1.1" 200 OK
/health status: 200
{'status': 'ok'}
INFO:     136.107.230.182:0 - "POST /predict HTTP/1.1" 200 OK

/predict status: 200
{'classe_prevista': 1, 'rotulo': 'falha', 'probabilidade_falha': 0.9923, 'recomendacao': 'Acionar equipe de manutenção.', 'entrada_recebida': {'temperatura_motor': 78.0, 'vibracao_motor': 4.5, 'corrente_motor': 9.8, 'pressao_sistema': 4.1, 'tempo_ciclo': 55.0}}


### Registro da evidência

Cole abaixo a URL gerada pelo ngrok:

________________________________________________________________________________

Cole abaixo o resultado do endpoint `/health`:

________________________________________________________________________________

Cole abaixo o resultado do endpoint `/predict`:

________________________________________________________________________________

Explique por que Colab + ngrok não deve ser usado como solução definitiva de produção:

________________________________________________________________________________

### Observações importantes

- Use `public_url + "/docs"` para acessar a documentação interativa do FastAPI.
- Use `public_url + "/predict"` para testar o endpoint de inferência.
- A URL pública muda a cada nova execução do ngrok.
- O Colab + ngrok é ótimo para demonstração e aula, mas não deve ser usado como produção.
- Em produção, utilize autenticação, HTTPS, logs, monitoramento, controle de versão e governança.

## 10. Documentação técnica

Complete a documentação técnica da API.

In [34]:
documentacao = """
# Documentação Técnica — API de Inferência de Falhas Industriais

## 1. Objetivo

________________________________________________________________________________

## 2. Modelo utilizado

- Tipo: __________________________________________________________________________
- Biblioteca: _____________________________________________________________________
- Entrada: _______________________________________________________________________
- Saída: _________________________________________________________________________

## 3. Endpoints

### GET /
________________________________________________________________________________

### GET /health
________________________________________________________________________________

### GET /metadata
________________________________________________________________________________

### POST /predict
________________________________________________________________________________

## 4. Fluxo de inferência

1. ________________________________________________________________________________
2. ________________________________________________________________________________
3. ________________________________________________________________________________
4. ________________________________________________________________________________
5. ________________________________________________________________________________

## 5. Cuidados técnicos

________________________________________________________________________________
"""

with open("DOCUMENTACAO_API_AULA18.md", "w", encoding="utf-8") as f:
    f.write(documentacao)

print("Documentação técnica gerada.")

Documentação técnica gerada.


In [72]:
documentacao = '''
# Documentacao Tecnica - API de Inferencia de Falhas Industriais
## 1. Objetivo
Disponibilizar um servico de predicao em tempo real que utiliza IA para identificar
potenciais falhas em motores industriais com base em dados de sensores.
## 2. Modelo utilizado
- Tipo: Rede Neural Artificial (Multi-Layer Perceptron - MLP)
- Biblioteca: Scikit-learn (MLPClassifier)
- Entrada: 5 variaveis numericas (temperatura, vibracao, corrente, pressao, tempo_ciclo)
- Saida: Classe binaria (0 = normal, 1 = falha) e probabilidade
## 3. Endpoints
### GET / → Boas-vindas e versao do modelo
### GET /health → Status do servidor e dos artefatos
### GET /metadata → Metadados completos do modelo
### POST /predict → Predicao de falha a partir de dados dos sensores
## 4. Fluxo de inferencia
1. Recebimento dos dados brutos via HTTP POST
2. Validacao pelo schema Pydantic
3. Organizacao das features na ordem do treino
4. Aplicacao do StandardScaler (mesmo do treinamento)
5. predict_proba() → probabilidade → classificacao
## 5. Cuidados tecnicos
- Usar SEMPRE o mesmo scaler_falhas.joblib gerado no treinamento
- Colab + ngrok e adequado apenas para demonstracao e aulas
- Em producao: HTTPS, autenticacao, logs, monitoramento e governanca
'''

with open ("DOCUMENTACAO API_AULA18.md", "w", encoding="utf-8") as f:
    f.write (documentacao)
print ("Documentacao tecnica gerada com sucesso.")

Documentacao tecnica gerada com sucesso.


## 11. Requirements e estrutura final

In [73]:
requirements_api = '''
fastapi
uvicorn
pydantic
pandas
numpy
scikit-learn
joblib
requests
'''.strip()

with open("requirements_api_aula18.txt", "w", encoding="utf-8") as f:
    f.write(requirements_api)

print(requirements_api)

fastapi
uvicorn
pydantic
pandas
numpy
scikit-learn
joblib
requests


Estrutura esperada:

```txt
projeto_api_ia/
├── app.py
├── artefatos_modelo/
│   ├── ____________________________
│   ├── ____________________________
│   └── ____________________________
├── DOCUMENTACAO_API_AULA18.md
├── requirements_api_aula18.txt
└── notebook_aula18.ipynb
```

## 12. Checklist da API funcional

In [75]:
checklist_api = {
    "modelo_treinado": True,
    "scaler_salvo": os.path.exists("artefatos_modelo/scaler_falhas.joblib"),
    "modelo_salvo": os.path.exists("artefatos_modelo/modelo_mlp_falhas.joblib"),
    "metadados_salvos": os.path.exists("artefatos_modelo/metadados_modelo.json"),
    "funcao_inferencia_testada": True,
    "arquivo_app_criado": os.path.exists("app.py"),
    "endpoint_predict_implementado": True,
    "documentacao_gerada": os.path.exists("DOCUMENTACAO_API_AULA18.md"),
    "requirements_criado": os.path.exists("requirements_api_aula18.txt")
}

pd.DataFrame([checklist_api]).T.rename(columns={0: "status"})

,status
modelo_treinado,True
scaler_salvo,False
modelo_salvo,True
metadados_salvos,True
funcao_inferencia_testada,True
arquivo_app_criado,True
endpoint_predict_implementado,True
documentacao_gerada,True
requirements_criado,True


## 13. Atividade prática avaliativa

Entregue uma API funcional contendo:

1. modelo treinado ou carregado;
2. scaler/preprocessador salvo;
3. metadados do modelo;
4. função local de inferência testada;
5. API criada com FastAPI;
6. endpoint `/predict`;
7. exemplo de requisição e resposta;
8. documentação técnica;
9. `requirements.txt`;
10. checklist de funcionamento.

## 14. Questões para reflexão

1. O que é uma API e por que ela é importante em projetos de IA?
2. Qual é a diferença entre treinamento e inferência?
3. Por que a API precisa usar o mesmo scaler do treinamento?
4. Qual é a função do FastAPI nesse projeto?
5. Quais cuidados seriam necessários para usar essa API em produção?

In [35]:
resposta_1 = """
Uma API (Application Programming Interface) é um conjunto de definições e protocolos que permite que diferentes softwares se comuniquem. Em projetos de IA, ela é crucial porque possibilita que outros sistemas (aplicativos web, mobile, sistemas corporativos) utilizem a inteligência de um modelo de IA treinado sem precisar conhecer seus detalhes internos, facilitando a integração de capacidades de IA em soluções maiores.
"""

resposta_2 = """
Treinamento é o processo de alimentar um modelo de Machine Learning com dados para que ele aprenda padrões e relações. Inferência (ou predição) é o uso desse modelo já treinado para fazer previsões ou tomar decisões sobre novos dados, nunca antes vistos pelo modelo.
"""

resposta_3 = """
A API precisa usar o mesmo scaler (ou qualquer transformação de pré-processamento) que foi utilizado durante o treinamento para garantir que os novos dados de entrada sejam transformados exatamente da mesma maneira que os dados de treinamento. Isso mantém a consistência na escala e distribuição dos dados, o que é fundamental para que o modelo faça previsões precisas. Se a escala for diferente, o modelo receberá dados em um formato para o qual não foi treinado, levando a um desempenho incorreto ou ruim.
"""

resposta_4 = """
Neste projeto, o FastAPI é utilizado para construir uma API web. Ele oferece um framework moderno e de alta performance para criar endpoints de API (como `/health`, `/predict`). Ele gerencia o roteamento, a validação de requisições/respostas (usando modelos Pydantic) e a serialização/desserialização de dados JSON, tornando eficiente a exposição das capacidades de inferência do modelo de IA via HTTP.
"""

resposta_5 = """
Para usar essa API em produção, seriam necessários os seguintes cuidados:
1. Segurança: Implementar autenticação e autorização (ex: chaves de API, OAuth) e HTTPS para comunicação criptografada.
2. Monitoramento: Implementar logs e monitoramento (ex: latência, taxas de erro, drift do modelo) para rastrear o desempenho e identificar problemas.
3. Escalabilidade: Implantar em infraestrutura escalável (ex: Kubernetes, funções em nuvem) para lidar com cargas variadas.
4. Confiabilidade: Implementar redundância, tolerância a falhas e tratamento de erros adequado.
5. Versionamento: Gerenciar versões da API e do modelo para garantir compatibilidade e atualizações suaves.
6. Governança: Estabelecer políticas para privacidade de dados, viés do modelo e uso ético da IA.
7. Infraestrutura: Mudar de Colab + ngrok para serviços de nuvem dedicados.
"""

print("Respostas registradas.")

Respostas registradas.


## 15. Critérios de avaliação

| Critério | Atingiu | Não atingiu |
|---|---|---|
| Estrutura fluxo de inferência corretamente | Organiza entrada, validação, pré-processamento, predição e resposta | Mistura etapas ou ignora pré-processamento |
| Integra modelo à aplicação proposta | Cria API com endpoint funcional de predição | Não consegue conectar modelo e aplicação |
| Documenta processo técnico | Descreve endpoints, entrada, saída e cuidados de uso | Não documenta ou documenta de forma insuficiente |
| Salva e carrega artefatos | Usa modelo, scaler e metadados salvos | Depende apenas de variáveis temporárias |
| Testa a API | Realiza ou descreve teste com requisição POST | Não valida funcionamento |
| Considera uso empresarial | Aponta segurança, autenticação, logs e governança | Trata API de teste como produção |